# Creating the document term matrix

## The Document-Term Matrix

Most quantitative text analysis methods do not operate directly on text. Instead, they require a **Document-Term Matrix (DTM)**.

In a DTM:
* each **row** represents a document;
* each **column** represents a unique lexical feature (word or normalized token);
* each **cell** contains the frequency with which that feature appears in the document.

For example:

| Document | economy | education | health | tax |
| -------- | ------: | --------: | -----: | --: |
| Party A  |      12 |         8 |      5 |   3 |
| Party B  |       4 |        15 |      9 |   1 |
| Party C  |      18 |         2 |      7 |  10 |

This numerical representation allows statistical and machine learning algorithms to analyze textual data.

However, constructing a useful DTM requires deciding **which lexical forms become columns**. Different representations of the same concept (e.g., plurals, verb conjugations, OCR errors, or missing accents) unnecessarily increase the number of columns and make the matrix sparser.


In [2]:
import pandas as pd

CORPUS_URL='https://github.com/Magallanes-at-UTDT/texts/raw/main/data/processed/CleanCorpus.csv'
corpus = pd.read_csv(CORPUS_URL)

corpus[["party", "n_words_clean"]]


,party,n_words_clean
0,AhoraNacion,50085
1,BuenGobierno,25876
2,FuerzaPopular,42623
3,JuntosPorElPeru,28899
4,Obras,5246
5,RenovacionPopular,5383


## Approach 1: the naive LLM prompt

The simplest thing anyone reaches for first: hand the LLM the cleaned text and
ask it to lemmatize it, with no schema, no format instructions, no guidance
on stopwords or POS filtering. Just:

> "Lemmatize this text"

We'll build the DTM from whatever comes back, and see what that decision
costs us downstream.


In [15]:
from pathlib import Path
from google import genai
from google.genai import types

MODEL = "gemini-3.6-flash"
API_KEY = Path("sometext.txt").read_text().strip()
client = genai.Client(api_key=API_KEY)

PROMPT_NAIVE = "Lemmatize this text:\n\n{text}"

def lemmatize_naive_llm(text):
    response = client.models.generate_content(
        model=MODEL,
        contents=PROMPT_NAIVE.format(text=text),
        config=types.GenerateContentConfig(temperature=0),
    )
    return response.text


In [16]:
# Test with a small sample first
test_text = "Los niños están corriendo rápidamente"
result = lemmatize_naive_llm(test_text)
print(f"Result: '{result}'")

Result: 'Here is the lemmatized version of the text, word by word and as a sequence:

### Word-by-Word Lemmatization:

* **Los** → **el** *(article: plural to masculine singular)*
* **niños** → **niño** *(noun: plural to singular)*
* **están** → **estar** *(verb: conjugated form to infinitive)*
* **corriendo** → **correr** *(verb: gerund form to infinitive)*
* **rápidamente** → **rápidamente** *(adverb: remains unchanged; base adjective is "rápido")*

---

### Lemmatized Sequence:
> **el niño estar correr rápidamente**'


**Now what?** The response above is continuous prose -- not a token list, not
JSON, not anything with an obvious column boundary. The model made its own
unstated decisions about lowercasing, punctuation, and stopwords, and we have
no record of what those decisions were. To get from this to a DTM, we still
have to write our own tokenization code -- asking the LLM to lemmatize didn't
remove that work, it just moved it to after the API call, with less control
over what the input to that tokenization step actually looks like.


In [5]:
import time

naive_llm_texts = {}
for row in corpus.itertuples():
    naive_llm_texts[row.party] = lemmatize_naive_llm(row.text_clean)
    time.sleep(1) # just in case

for party, text in naive_llm_texts.items():
    print(f"{party}: {len(text)} characters returned")

import time

# Run the bare-prompt call once per party, on the FULL text_clean this time
# (not the 1000-character sample above). Some of these documents are large
# (AhoraNacion's text_clean is ~340K characters) -- with no response_schema
# constraining the output, a long input increases the odds of the model's
# response getting cut off by its own output-token limit, and there's no
# structured field here to check a finish_reason against before trusting the
# result, unlike the JSON-schema approach in the earlier lemmatization
# notebook. Worth keeping an eye on the character counts printed below.
naive_llm_texts = {}
for row in corpus.itertuples():
    naive_llm_texts[row.party] = lemmatize_naive_llm(row.text_clean)
    time.sleep(1)  # stay polite to the API between the 6 calls

# Quick sanity check before moving on: does the returned length look at all
# proportional to the input length? A response that's suspiciously short
# relative to its party's n_words_clean is a sign of truncation, not just
# normal lemmatization shrinkage.
for party, text in naive_llm_texts.items():
    print(f"{party}: {len(text)} characters returned")


AhoraNacion: 120516 characters returned
BuenGobierno: 180695 characters returned
FuerzaPopular: 175631 characters returned
JuntosPorElPeru: 50056 characters returned
Obras: 34784 characters returned
RenovacionPopular: 37530 characters returned


In [9]:
naive_llm_texts

{'AhoraNacion': 'A continuación se presenta el texto íntegramente lematizado (las palabras flexionadas —verbos, sustantivos, adjetivos, determinantes y pronombres— han sido reducidas a su forma base o de diccionario: verbo en infinitivo, sustantivos/adjetivos en masculino singular, artículos en forma canónica *el/un*, etc.):\n\n***\n\nAHORA NACIÓN "todo el poder a el región para conquistar el mercado de el mundo" ÍNDICE I. PRESENTACIÓN II. IDEARIO: II.1. PRINCIPIO: II.2. OBJETIVO: II.3. VISIÓN DE EL PARTIDO: II.4. VALOR DE EL AHORA NACIÓN: III. VISIÓN DE EL PLAN DE GOBIERNO: IV. ESTRATEGIA: IV.1. DIMENSIÓN SOCIAL IV.1.1. PROPUESTA PARA EL SALUD IV.1.2. PROPUESTA PARA EL EDUCACIÓN IV.1.3. PROPUESTA PARA EL VIVIENDA IV.1.4. PROPUESTA PARA EL AGUA Y SANEAMIENTO IV.1.5. PROPUESTA PARA EL FOMENTO DE EL EMPLEO IV.1.6. PROPUESTA PARA INCLUSIÓN SOCIAL IV.1.7. PROPUESTA PARA CULTURA IV.2. DIMENSIÓN INSTITUCIONAL IV.2.1. PROPUESTA PARA EL SEGURIDAD CIUDADANO IV.2.2. PROPUESTA PARA EL JUSTICIA Y 

In [11]:
# Dictionary: {'Democrat': 'processed text', 'Republican': 'processed text', ...}
pd.DataFrame(list(naive_llm_texts.items()), columns=['party', 'naive_llm_text'])

,party,naive_llm_text
0,AhoraNacion,A continuación se presenta el texto íntegramen...
1,BuenGobierno,A continuación se presenta el texto completame...
2,FuerzaPopular,A continuación se presenta el texto completo l...
3,JuntosPorElPeru,A continuación se presenta la lemmatización de...
4,Obras,Aquí tienes el texto completo lemmatizado (sus...
5,RenovacionPopular,Aquí tienes el texto lemmatizado (verbos en in...


In [10]:
# Map the dictionary back to your original DataFrame
corpus['party'].map(naive_llm_texts)

0    A continuación se presenta el texto íntegramen...
1    A continuación se presenta el texto completame...
2    A continuación se presenta el texto completo l...
3    A continuación se presenta la lemmatización de...
4    Aquí tienes el texto completo lemmatizado (sus...
5    Aquí tienes el texto lemmatizado (verbos en in...
Name: party, dtype: object

## Building the DTM from the naive output

Consistent with how naive this approach has been so far, we tokenize with the
simplest possible method: lowercase, split on non-letters, nothing else -- no
stopword list, no POS filtering, no lemmatization double-check. This is
`CountVectorizer`'s own default behavior, so we don't even have to write it
ourselves.


In [6]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_naive = CountVectorizer(lowercase=True)
dtm_naive = vectorizer_naive.fit_transform(
    [naive_llm_texts[party] for party in corpus["party"]]
)
dtm_naive_df = pd.DataFrame(
    dtm_naive.toarray(),
    index=corpus["party"],
    columns=vectorizer_naive.get_feature_names_out(),
)
dtm_naive_df


,00,000,01,011,015,0270,03,030,083,0e5,...,íntegramente,íntegro,óptica,óptico,órbita,órgano,último,única,único,útil
party,,,,,,,,,,,,,,,,,,,,,
AhoraNacion,0,55,0,0,1,0,1,0,1,0,...,1,4,0,0,0,1,4,0,3,0
BuenGobierno,0,11,2,0,0,0,0,0,0,1,...,2,2,1,4,1,6,1,0,18,1
FuerzaPopular,0,33,0,1,0,0,0,1,0,0,...,0,0,0,3,0,0,8,2,16,0
JuntosPorElPeru,0,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,2,1,1,4,0
Obras,0,12,0,0,0,1,0,0,0,0,...,0,0,0,0,0,1,4,0,6,1
RenovacionPopular,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,4,0,0,0


## How sparse did we end up, despite the lemmatization step?

This is the number to compare against the guided (spaCy) notebook later --
if a bare LLM lemmatization call didn't actually reduce sparsity relative to
skipping lemmatization entirely, that's the concrete evidence for why the
intro section's point about redundant lexical forms matters.


In [7]:
n_docs, n_terms = dtm_naive_df.shape
n_nonzero = (dtm_naive_df > 0).values.sum()
sparsity = 1 - (n_nonzero / (n_docs * n_terms))

print(f"Vocabulary size (columns): {n_terms}")
print(f"Sparsity: {sparsity:.2%} of cells are zero")
dtm_naive_df.sum(axis=0).sort_values(ascending=False).head(20)


Vocabulary size (columns): 5768
Sparsity: 63.97% of cells are zero


de            7524
el            7494
en            2038
con           1141
para          1019
que            836
uno            681
del            623
nacional       606
ser            590
meta           498
por            428
él             405
público        390
sistema        388
un             339
desarrollo     324
su             313
como           291
perú           266
dtype: int64

## TF-IDF, derived from the same counts

TF-IDF is a reweighting of the counts we already have, not a separate
pipeline -- `TfidfTransformer` takes the DTM we just built rather than
re-tokenizing the text a second time. Keep in mind: several standard
comparative-text methods (Wordfish, Wordscores, LDA topic modeling, keyness
statistics) need the raw-count DTM specifically and would be misapplied on
TF-IDF values -- TF-IDF is the derived view for similarity/clustering tasks,
not a replacement for the counts.


In [8]:
from sklearn.feature_extraction.text import TfidfTransformer

tfidf_transformer = TfidfTransformer()
tfidf_naive = tfidf_transformer.fit_transform(dtm_naive)
tfidf_naive_df = pd.DataFrame(
    tfidf_naive.toarray(),
    index=dtm_naive_df.index,
    columns=dtm_naive_df.columns,
)
tfidf_naive_df


,00,000,01,011,015,0270,03,030,083,0e5,...,íntegramente,íntegro,óptica,óptico,órbita,órgano,último,única,único,útil
party,,,,,,,,,,,,,,,,,,,,,
AhoraNacion,0.00000,0.020904,0.000000,0.000000,0.000856,0.000000,0.000856,0.000000,0.000856,0.000000,...,0.000702,0.002808,0.000000,0.000000,0.000000,0.000439,0.001520,0.000000,0.001316,0.000000
BuenGobierno,0.00000,0.004032,0.001652,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000826,...,0.001354,0.001354,0.000826,0.002709,0.000826,0.002538,0.000367,0.000000,0.007615,0.000677
FuerzaPopular,0.00000,0.008959,0.000000,0.000612,0.000000,0.000000,0.000000,0.000612,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.001505,0.000000,0.000000,0.002172,0.001003,0.005013,0.000000
JuntosPorElPeru,0.00000,0.002082,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.002403,0.001041,0.001923,0.004806,0.000000
Obras,0.00000,0.017674,0.000000,0.000000,0.000000,0.003318,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.001700,0.005891,0.000000,0.010199,0.002721
RenovacionPopular,0.00261,0.001159,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.001337,0.004634,0.000000,0.000000,0.000000


## Save both matrices

For comparison against the guided (spaCy) notebook's DTM once it's built.


In [ ]:
dtm_naive_df.to_csv("DTM_llm_naive_counts.csv")
tfidf_naive_df.to_csv("DTM_llm_naive_tfidf.csv")
print("Saved: DTM_llm_naive_counts.csv, DTM_llm_naive_tfidf.csv")
